In [1]:
%pip install pypdf sentence-transformers

  Using cached setuptools-84.0.0-py3-none-any.whl.metadata (6.6 kB)
   ---------------------------------------- 0.0/740.6 kB ? eta -:--:--
   --------------------------------------- 740.6/740.6 kB 29.9 MB/s eta 0:00:00
   ---------------------------------------- 0.0/846.4 kB ? eta -:--:--
   --------------------------------------- 846.4/846.4 kB 36.5 MB/s eta 0:00:00
   ---------------------------------------- 0.0/8.3 MB ? eta -:--:--
   ---------------------------------------- 8.3/8.3 MB 64.0 MB/s eta 0:00:00
   ---------------------------------------- 0.0/36.7 MB ? eta -:--:--
   ----------------- ---------------------- 15.7/36.7 MB 82.7 MB/s eta 0:00:01
   ------------------------------------ --- 33.0/36.7 MB 84.0 MB/s eta 0:00:01
   ---------------------------------------- 36.7/36.7 MB 75.2 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.9 MB ? eta -:--:--
   ---------------------------------------- 2.9/2.9 MB 55.5 MB/s eta 0:00:00
   ----------------------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] T

In [1]:
from agents.db import get_connection
conn = get_connection()

In [5]:
import glob, os
from pypdf import PdfReader

def chunk_text(text, size=1000, overlap=200):
    chunks, start = [], 0
    while start < len(text):
        chunks.append(text[start:start + size])
        start += size - overlap
    return chunks

docs = []
for path in glob.glob("policy_docs/*.pdf"):
    text = " ".join((page.extract_text() or "") for page in PdfReader(path).pages)
    text = " ".join(text.split())
    for i, ch in enumerate(chunk_text(text)):
        docs.append((os.path.basename(path), i, ch))

print(len(docs), "chunks from", len({d[0] for d in docs}), "PDFs")

64 chunks from 6 PDFs


In [6]:
print("Files in policy_docs:")
for path in glob.glob("policy_docs/*"):
    print("  ", path)

print("\nText found in each PDF:")
for path in glob.glob("policy_docs/*.pdf"):
    text = " ".join((page.extract_text() or "") for page in PdfReader(path).pages)
    print("  ", os.path.basename(path), "→", len(text.strip()), "characters")

Files in policy_docs:
   policy_docs\access_in_motion.pdf
   policy_docs\boston_closure_shuttle_policy.pdf
   policy_docs\mbta_accessibility_overview.pdf
   policy_docs\mbta_elevator_alerts.pdf
   policy_docs\sources.md
   policy_docs\station_downtown_crossing.pdf
   policy_docs\the_ride.pdf

Text found in each PDF:
   access_in_motion.pdf → 17908 characters
   boston_closure_shuttle_policy.pdf → 8274 characters
   mbta_accessibility_overview.pdf → 6737 characters
   mbta_elevator_alerts.pdf → 6548 characters
   station_downtown_crossing.pdf → 1768 characters
   the_ride.pdf → 7875 characters


In [7]:
import json
from agents.retriever import embed

cur = conn.cursor()
cur.execute("TRUNCATE TABLE TRANSITWISE_DB.RAG.POLICY_CHUNKS")

for source, n, ch in docs:
    cur.execute("""
        INSERT INTO TRANSITWISE_DB.RAG.POLICY_CHUNKS
        SELECT %s, %s, %s, PARSE_JSON(%s)::ARRAY::VECTOR(FLOAT, 384)
    """, (source, n, ch, json.dumps(embed(ch))))

print("Loaded", len(docs), "chunks into Snowflake")

c:\Users\agast\AppData\Local\Python\pythoncore-3.12-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6717.04it/s]


Loaded 64 chunks into Snowflake


In [9]:
from agents.retriever import search_policies

for r in results:
    print(f"Score: {r['score']}   Source: {r['source']}")
    print(r["chunk"][:400])
    print("-" * 60)

Score: 0.684   Source: mbta_elevator_alerts.pdf
form to paid lobby) is unavailable due to maintenance. Downtown Crossing Downtown Crossing Elevator Closure ONGOING 10/14/26, 12:47 AM Alerts | MBTA https://www.mbta.com/alerts/access 3/8 Downtown Crossing Elevator 869 Alewife platform to Summer Street Concourse) is unavailable due to maintenance through Tuesday, December 1. ⌄ Government Center Government Center Escalator Closure Government Cente
------------------------------------------------------------
Score: 0.643   Source: mbta_elevator_alerts.pdf
oadway Broadway Escalator Closure ONGOING Broadway Escalator 367 Platform to paid lobby) is unavailable due to maintenance. Central Avenue Central Avenue Access Issue ONGOING The Central Avenue mini-high platform Ashmont side) is out of service. ⌄ Courthouse Courthouse Escalator Closure ONGOING Courthouse Escalator 415 Airport/Design Center platform to lobby) unavailable due to ongoing construc
-----------------------------------------